# 迁移学习：站在巨人肩膀上
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：09_计算机视觉 | 源文件：迁移学习_图像.py | 核心功能：预训练模型微调、特征提取、冻结层

## 概述

迁移学习用在大数据集（如 ImageNet）上预训练的模型，微调到目标任务。两种策略：特征提取（冻结预训练层）和微调（部分或全部解冻）。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

## 数学原理

### 1. 迁移学习的理论基础

迁移学习利用源域 $D_S$ 上学到的知识帮助目标域 $D_T$ 的学习。设源域预训练模型为 $f_S$，目标域模型为 $f_T$。

**表示迁移假设**：深度网络的低层学到通用特征（边缘、纹理），高层学到任务特定特征。

$$f_T(x) = W_T \cdot \phi_S(x)$$

其中 $\phi_S$ 是预训练网络的特征提取器（冻结），$W_T$ 是新任务的分类头。

### 2. 策略一：特征提取（Feature Extraction）

冻结预训练网络的所有层，只训练新增的分类头：

$$\theta^* = \arg\min_\theta \sum_{i} L(W_\theta \cdot \phi_{frozen}(x_i), y_i)$$

只有 $\theta$（分类头参数）参与梯度更新，$\phi_{frozen}$ 的参数不变。

代码中 `requires_grad = False` 冻结参数。

### 3. 策略二：微调（Fine-Tuning）

解冻部分或全部预训练层，用小学习率联合训练：

$$\theta^*, \phi^* = \arg\min_{\theta, \phi} \sum_{i} L(W_\theta \cdot \phi_\phi(x_i), y_i)$$

微调使用较小的学习率 $\eta_{ft} \ll \eta_{pre}$，避免破坏预训练权重。

代码中冻结浅层、解冻深层的策略（逐层解冻）。

### 4. 学习率策略

**分层学习率**：不同层使用不同学习率

$$\eta_l = \eta_{base} \cdot \alpha^{L-l}$$

其中 $l$ 是层索引（浅层 $l$ 小），$\alpha < 1$ 使浅层学习率更低。

### 5. 迁移的条件

迁移效果取决于源域和目标域的**相似度**：

$$\text{Transfer Gain} \propto \text{similarity}(D_S, D_T)$$

| 情况 | 策略 |
|------|------|
| 小数据 + 相似域 | 特征提取 |
| 小数据 + 不同域 | 特征提取或微调浅层 |
| 大数据 + 相似域 | 微调全部层 |
| 大数据 + 不同域 | 微调全部层，可能需要更多轮次 |

### 6. ResNet18 的结构与冻结

ResNet18 包含多个残差块，代码中冻结策略：

| 层 | 参数 | 冻结策略 |
|----|------|----------|
| conv1 + bn1 | $64 \times 3 \times 7 \times 7$ | 特征提取时冻结 |
| layer1-4 | 残差块 | 微调时可逐层解冻 |
| fc | $512 \times C$ | 始终重新训练 |

替换 `fc` 层：$W_{fc} \in \mathbb{R}^{C \times 512} \to W'_{fc} \in \mathbb{R}^{C_{new} \times 512}$

### 7. 预训练权重的统计正则化

预训练权重 $W_{pre}$ 提供了良好的初始化，隐式正则化：

$$\|\theta - \theta_{pre}\|^2 \leq \epsilon$$

微调时的学习率越小，$\epsilon$ 越小，正则化效果越强。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `models.resnet18(weights=DEFAULT)` | 加载预训练参数 $\theta_{pre}$ |
| `param.requires_grad = False` | 冻结参数，不计算梯度 |
| `model.fc = nn.Linear(512, 10)` | 替换分类头 $W' \in \mathbb{R}^{10 \times 512}$ |
| `lr=1e-3`（特征提取） | 只训练分类头，较大 LR |
| `lr=1e-4`（微调） | 解冻层用小 LR，避免破坏预训练权重 |
| `Subset(dataset, indices)` | 用子集模拟小数据场景 |
| `model.fc.in_features = 512` | 预训练模型的特征维度 |

### 策略一：特征提取

运行 策略一：特征提取 的代码，观察算法在该环节的行为。

In [ ]:

def build_feature_extraction_model(num_classes=10):
    """
    特征提取：冻结预训练 ResNet18 的所有层，只替换最后的全连接层。

    原理：
    - 预训练层的权重完全冻结，不参与梯度更新
    - 只有新增的分类头会被训练
    - 速度快，适合小数据集

    返回:
        model: 模型
        params_to_train: 需要训练的参数
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # 冻结所有参数
    for param in model.parameters():
        param.requires_grad = False

    # 替换最后的全连接层（ResNet18 原始输出是 1000 类）
    in_features = model.fc.in_features  # 512
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )

    # 只有新加的层需要训练
    params_to_train = filter(lambda p: p.requires_grad, model.parameters())

    return model, params_to_train

### 策略二：微调

运行 策略二：微调 的代码，观察算法在该环节的行为。

In [ ]:

def build_finetune_model(num_classes=10, freeze_until=None):
    """
    微调：解冻部分或全部预训练层，用较小学习率训练。

    参数:
        num_classes: 分类数
        freeze_until: 冻结到第几层（None=不解冻，int=冻结前 N 层）

    原理：
    - 解冻预训练层，允许它们根据新任务调整权重
    - 使用很小的学习率（如 1e-4），避免破坏预训练特征
    - 通常分类头用更大学习率，预训练层用更小学习率

    返回:
        model: 模型
        params_group: 参数组（不同学习率）
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # 替换分类头
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )

    if freeze_until is not None:
        # 冻结前 N 层
        children = list(model.children())
        for i, child in enumerate(children):
            if i < freeze_until:
                for param in child.parameters():
                    param.requires_grad = False

    # 将参数分为两组：预训练层（小学习率）和分类头（大学习率）
    pretrained_params = [p for p in model.parameters() if p.requires_grad and p.shape[0] != num_classes]
    classifier_params = list(model.fc.parameters())

    params_group = [
        {"params": pretrained_params, "lr": 1e-4},
        {"params": classifier_params, "lr": 1e-3},
    ]

    return model, params_group

### 数据加载

运行 数据加载 的代码，观察算法在该环节的行为。

In [ ]:

def get_dataloaders(batch_size=64, subset_size=None):
    """
    加载 CIFAR-10 数据集。

    参数:
        batch_size: 批量大小
        subset_size: 使用子集的大小（None=使用全部数据，用于加速演示）
    """
    transform_train = transforms.Compose([
# --- 继续 ---
        transforms.Resize(64),  # CIFAR-10 原始 32×32，放大到 64 适配 ResNet
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.Resize(64),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
    test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

    # 可选：使用子集加速演示
    if subset_size is not None:
        train_dataset = Subset(train_dataset, range(min(subset_size, len(train_dataset))))
        test_dataset = Subset(test_dataset, range(min(subset_size // 5, len(test_dataset))))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, test_loader

### 训练和评估

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:

def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch"""
    model.train()
    total_loss, correct, total = 0, 0, 0
    for data, target in loader:
# --- 继续 ---
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
# --- 继续 ---
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        correct += (output.argmax(1) == target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, device):
    """评估模型准确率"""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
# --- 循环处理 ---
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            correct += (output.argmax(1) == target).sum().item()
            total += target.size(0)
# --- 返回结果 ---
    return correct / total

### 主流程

运行 主流程 的代码，观察算法在该环节的行为。

In [ ]:

def main():
    print("=" * 50)
    print("迁移学习 — 特征提取 vs 微调")
    print("=" * 50)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 超参数
    batch_size = 32
    epochs = 3
    subset_size = 2000  # 使用小数据集加速演示

    # 加载数据
    print(f"\n加载 CIFAR-10 数据集（使用 {subset_size} 样本子集）...")
    train_loader, test_loader = get_dataloaders(batch_size, subset_size)
    print(f"训练集大小: {len(train_loader.dataset)}  测试集大小: {len(test_loader.dataset)}")
    print(f"CIFAR-10 类别: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck")

    results = {}

    # --- 策略一：特征提取 ---
    print(f"\n{'='*50}")
    print("策略一: 特征提取 (Feature Extraction)")
    print("  - 冻结所有预训练层")
    print("  - 只训练新的分类头")
    print(f"{'='*50}")

    model_fe, params_fe = build_feature_extraction_model(num_classes=10)
    model_fe = model_fe.to(device)

    # 统计可训练参数
    trainable_fe = sum(p.numel() for p in params_fe)
    total_fe = sum(p.numel() for p in model_fe.parameters())
    print(f"  总参数: {total_fe:,}  可训练参数: {trainable_fe:,} ({trainable_fe/total_fe*100:.1f}%)")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(params_fe, lr=1e-3)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model_fe, train_loader, criterion, optimizer, device)
        test_acc = evaluate(model_fe, test_loader, device)
        print(f"  Epoch {epoch}: 损失={train_loss:.4f}, 训练准确率={train_acc*100:.2f}%, 测试准确率={test_acc*100:.2f}%")

    results["特征提取"] = test_acc

    # --- 策略二：微调 ---
    print(f"\n{'='*50}")
    print("策略二: 微调 (Fine-Tuning)")
    print("  - 冻结 ResNet 的前 6 层（conv1 + layer1-3）")
    print("  - 解冻 layer4 和分类头")
    print("  - 分类头学习率 1e-3，预训练层学习率 1e-4")
# --- 输出结果 ---
    print(f"{'='*50}")

    model_ft, params_ft = build_finetune_model(num_classes=10, freeze_until=6)
    model_ft = model_ft.to(device)

    trainable_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
    total_ft = sum(p.numel() for p in model_ft.parameters())
    print(f"  总参数: {total_ft:,}  可训练参数: {trainable_ft:,} ({trainable_ft/total_ft*100:.1f}%)")

    optimizer_ft = optim.Adam(params_ft, lr=1e-3)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model_ft, train_loader, criterion, optimizer_ft, device)
        test_acc = evaluate(model_ft, test_loader, device)
        print(f"  Epoch {epoch}: 损失={train_loss:.4f}, 训练准确率={train_acc*100:.2f}%, 测试准确率={test_acc*100:.2f}%")

    results["微调"] = test_acc

    # --- 从零训练作为基线 ---
    print(f"\n{'='*50}")
    print("基线: 从零训练 (Training from Scratch)")
    print("  - 不使用预训练权重")
    print(f"{'='*50}")

    model_scratch = models.resnet18(weights=None)
    model_scratch.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, 10))
    model_scratch = model_scratch.to(device)

    optimizer_scratch = optim.Adam(model_scratch.parameters(), lr=1e-3)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model_scratch, train_loader, criterion, optimizer_scratch, device)
        test_acc = evaluate(model_scratch, test_loader, device)
        print(f"  Epoch {epoch}: 损失={train_loss:.4f}, 训练准确率={train_acc*100:.2f}%, 测试准确率={test_acc*100:.2f}%")

    results["从零训练"] = test_acc

    # --- 结果对比 ---
    print(f"\n{'='*50}")
    print("三种策略对比:")
    print(f"{'='*50}")
    for name, acc in results.items():
        print(f"  {name:　<8s}: 测试准确率 {acc*100:.2f}%")

    # --- 迁移学习指导 ---
    print("\n--- 迁移学习策略选择指南 ---")
    print("  场景                          推荐策略")
    print("  数据少 (<1000) + 任务相似      → 特征提取")
    print("  数据少 (<1000) + 任务差异大    → 微调（冻结浅层）")
    print("  数据多 (>10000) + 任务相似     → 微调（解冻深层）")
# --- 输出结果 ---
    print("  数据多 (>10000) + 任务差异大   → 微调（解冻全部）")
    print("\n--- 常用预训练模型 ---")
    print("  ResNet: 经典残差网络，适合大多数视觉任务")
    print("  VGG: 结构简单，特征提取能力强，参数量大")
    print("  EfficientNet: 效率与精度平衡，适合移动端")
# --- 输出结果 ---
    print("  ViT: Vision Transformer，适合大规模数据集")


if __name__ == "__main__":
    main()

## 关键代码解释

```python
model = torchvision.models.resnet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False  # 冻结
model.fc = nn.Linear(512, num_classes)  # 替换最后一层
```

## 注意事项

1. 小数据集优先冻结大部分层
2. 学习率要比从头训练小（避免破坏预训练特征）
3. 域差异大时可能需要微调更多层

## 总结与延伸

以上代码展示了 **迁移学习 图像** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Foundation Models**：大规模预训练模型的时代
- **Prompt Tuning**：不改变模型参数的微调
- **Domain Adaptation**：跨域迁移学习
